# SONA Kaggle Run-All Notebook

Use this after uploading the refreshed `sona.zip` as a new version of your `sona-repo` Kaggle dataset.

Before running:
1. Add the updated `sona-repo` dataset.
2. Add your VoiceBank-DEMAND 16 kHz dataset containing `clean_trainset_wav`, `noisy_trainset_wav`, `clean_testset_wav`, and `noisy_testset_wav`.
3. Set Accelerator to GPU T4 and Internet to On.
4. Click Run All.


In [ ]:
# 1) Install metric dependencies. Torch/torchaudio are preinstalled on Kaggle.
%pip install -q pesq pystoi ptflops

import torch
import torchaudio

print('torch', torch.__version__)
print('torchaudio', torchaudio.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Turn on GPU accelerator before running the full benchmark.'

In [ ]:
# 2) Auto-detect repo and VoiceBank-DEMAND folders among attached Kaggle datasets.
import glob
import os
from pathlib import Path

def find_repo():
    hits = glob.glob('/kaggle/input/**/scripts/kaggle_run.py', recursive=True)
    if not hits:
        return None
    # Prefer a dataset path containing sona, but accept the first valid repo.
    hits = sorted(hits, key=lambda p: ('sona' not in p.lower(), len(p)))
    return str(Path(hits[0]).parents[1])

def has_vbd_layout(root):
    root = Path(root)
    needed = ['clean_trainset_wav', 'noisy_trainset_wav', 'clean_testset_wav', 'noisy_testset_wav']
    return all((root / name).is_dir() for name in needed)

def find_vbd():
    hits = glob.glob('/kaggle/input/**/noisy_trainset_wav', recursive=True)
    candidates = []
    for hit in hits:
        root = str(Path(hit).parent)
        if has_vbd_layout(root):
            candidates.append(root)
    if not candidates:
        return None
    return sorted(candidates, key=len)[0]

REPO_DIR = find_repo()
VBD_ROOT = find_vbd()
OUT_DIR = '/kaggle/working/results'

# If auto-detect fails, uncomment and set these manually from the Kaggle right panel.
# REPO_DIR = '/kaggle/input/sona-repo'
# VBD_ROOT = '/kaggle/input/voicebank-demand-16k'

assert REPO_DIR, 'sona-repo not found. Attach the updated sona-repo dataset.'
assert VBD_ROOT, 'VoiceBank-DEMAND not found. Attach a dataset with all four clean/noisy train/test folders.'

print('REPO_DIR =', REPO_DIR)
print('VBD_ROOT =', VBD_ROOT)
print('OUT_DIR  =', OUT_DIR)

# Make sure the uploaded repo is the new one with supervised-online support.
run_py = Path(REPO_DIR) / 'scripts' / 'kaggle_run.py'
txt = run_py.read_text(encoding='utf-8')
assert '--online-supervised' in txt, 'This sona-repo is old. Upload the refreshed sona.zip as a new dataset version.'
print('Repo check passed: supervised-online flags are present.')

In [ ]:
# 3) Scenario A retry settings.
# This is the fast decision-gate run: enough to verify attention_online before the full paper run.
LIMIT_TRAIN = 2000
LIMIT_TEST = 300
EPOCHS = 40
RL_EPOCHS = 40
SUP_MODELS = 'mask_gru,mask_tcn,attention'
ONLINE_SUP = 'attention,mask_gru'
ONLINE_SUP_ADAPTER = 'lowrank'
ONLINE_ANCHOR = 0.05
ABLATE = False

print(dict(
    LIMIT_TRAIN=LIMIT_TRAIN,
    LIMIT_TEST=LIMIT_TEST,
    EPOCHS=EPOCHS,
    RL_EPOCHS=RL_EPOCHS,
    SUP_MODELS=SUP_MODELS,
    ONLINE_SUP=ONLINE_SUP,
    ONLINE_SUP_ADAPTER=ONLINE_SUP_ADAPTER,
    ONLINE_ANCHOR=ONLINE_ANCHOR,
    ABLATE=ABLATE,
))

In [ ]:
# 4) Run the full benchmark and stream logs live.
import subprocess
import sys

cmd = [
    sys.executable, f'{REPO_DIR}/scripts/kaggle_run.py',
    '--repo', REPO_DIR,
    '--vbd-root', VBD_ROOT,
    '--out', OUT_DIR,
    '--limit-train', str(LIMIT_TRAIN),
    '--epochs', str(EPOCHS),
    '--rl-epochs', str(RL_EPOCHS),
    '--sup-models', SUP_MODELS,
    '--online-supervised', ONLINE_SUP,
    '--online-supervised-adapter', ONLINE_SUP_ADAPTER,
    '--online-anchor-weight', str(ONLINE_ANCHOR),
]
if LIMIT_TEST:
    cmd += ['--limit-test', str(LIMIT_TEST)]
if ABLATE:
    cmd += ['--ablate-adapters']

print(' '.join(cmd), '\n')
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
assert proc.returncode == 0, f'kaggle_run.py exited {proc.returncode}'

In [ ]:
# 5) Show saved summary table and the key rows to paste back to Codex.
import pandas as pd
from pathlib import Path

summary_md = Path(OUT_DIR) / 'summary.md'
summary_csv = Path(OUT_DIR) / 'summary.csv'

print(summary_md.read_text(encoding='utf-8'))
df = pd.read_csv(summary_csv)
display(df)

print('\nKey comparison rows:')
wanted = ['neural_attention', 'attention_online', 'neural_mask_gru', 'mask_gru_online', 'rl_frozen', 'rl_online']
display(df[df['method'].isin(wanted)])

In [ ]:
# 6) Bundle all trained checkpoints and result tables into one downloadable zip.
import glob
import os
import shutil
from IPython.display import FileLink, display

ART = '/kaggle/working/sona_artifacts'
shutil.make_archive(ART, 'zip', OUT_DIR)
print(f'Wrote {ART}.zip ({os.path.getsize(ART + ".zip") / 1e6:.1f} MB)')
print('\nContents:')
for f in sorted(glob.glob(OUT_DIR + '/**', recursive=True)):
    if os.path.isfile(f):
        print('  ', os.path.relpath(f, OUT_DIR), f'({os.path.getsize(f) / 1e3:.0f} KB)')
display(FileLink(ART + '.zip'))